# Final revision — Step 1

Run this notebook from this copied project folder, or set `NETWORK_ANALYSIS_LOCAL_ROOT` to its absolute path. Run Steps 01–07 in separate kernels. Existing outputs in this copy may be overwritten; the source repository is untouched.

The approved hypotheses, thresholds, seeds, null ensembles and removal protocol are retained. Cleared outputs avoid presenting historical execution as validation of revised code. Step 04 is the full historical generation workflow and is computationally expensive; the supplied cached ensembles allow Steps 05–07 to run without regenerating it. Consult `VALIDATION.md` for exactly what was rerun.


1. Connect Google Drive and import libraries

In [1]:
import os

import pandas as pd
import numpy as np
from pathlib import Path

# Notice the trailing space after 'Network Analysis '
LOCAL_PROJECT_ROOT = os.environ.get("NETWORK_ANALYSIS_LOCAL_ROOT")
if LOCAL_PROJECT_ROOT:
    BASE = Path(LOCAL_PROJECT_ROOT).expanduser().resolve() / "Data"
elif (Path.cwd() / "Data").is_dir():
    BASE = Path.cwd() / "Data"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/Network Analysis /Data")
RAW = BASE / "raw"
PROCESSED = BASE / "Processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW.exists())
print("Files found:", [p.name for p in RAW.glob("*.csv")])

Raw folder exists: True
Files found: ['April 2025.csv', 'June 2025.csv', 'May 2025.csv']


2 . Load and validate the raw datasets

In [2]:

files = {
    4: "April 2025.csv",
    5: "May 2025.csv",
    6: "June 2025.csv"
}

frames = []

for month, filename in files.items():
    df = pd.read_csv(RAW / filename, low_memory=False)

    # Check that each file contains the expected period
    assert (df["YEAR"] == 2025).all()
    assert (df["MONTH"] == month).all()

    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

    assert (df["FL_DATE"].dt.year == 2025).all()
    assert (df["FL_DATE"].dt.month == month).all()

    print(f"{filename}: {len(df):,} records")
    frames.append(df)

# All monthly files should have the same columns
assert all(set(f.columns) == set(frames[0].columns)
           for f in frames)

flights = pd.concat(frames, ignore_index=True)

print("\nTotal records:", f"{len(flights):,}")
print("Columns:", len(flights.columns))


Network_Analysis_01_Data_Preparation (1).ipynb:cell4:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
April 2025.csv: 644,084 records
Network_Analysis_01_Data_Preparation (1).ipynb:cell4:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
May 2025.csv: 667,586 records
Network_Analysis_01_Data_Preparation (1).ipynb:cell4:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
June 2025.csv: 674,179 records

Total records: 1,985,849
Columns: 19


Filter the airlines

In [3]:

AIRLINES = ["AA", "DL", "WN"]

flights = flights[
    flights["MKT_UNIQUE_CARRIER"].isin(AIRLINES)
].copy()

print("Selected flight records:", f"{len(flights):,}")

display(
    flights.groupby(["MKT_UNIQUE_CARRIER", "MONTH"])
           .size()
           .unstack(fill_value=0)
)


Selected flight records: 1,292,785


MONTH,4,5,6
MKT_UNIQUE_CARRIER,,,
AA,166972,173676,173558
DL,133985,141809,142996
WN,117622,119819,122348


4. Validate our network construction assumptions

**Question	Decision**


*   What represents a node?	A distinct airport ID
*   When does an edge exist?	At least one scheduled nonstop flight
*   Is the network directed?	Yes, origin → destination
*   What is the edge weight?	Total number of scheduled flights
*   Do we include canceled flights?	Yes, because we study scheduled connectivity
*   Do we use operating or marketing carrier?	Marketing carrier, including regional partners
*   Do we include diverted flights?	Yes, using their originally scheduled destination

Check missing values and flags

In [4]:

essential = [
    "FL_DATE",
    "MKT_UNIQUE_CARRIER",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "FLIGHTS"
]

print("Missing essential values:")
print(flights[essential].isna().sum())

print("\nFlight count values:")
print(flights["FLIGHTS"].value_counts(dropna=False))

print("\nCancellation summary:")
display(
    flights.groupby("MKT_UNIQUE_CARRIER")
    .agg(
        records=("FLIGHTS", "size"),
        scheduled_flights=("FLIGHTS", "sum"),
        cancelled=("CANCELLED", "sum"),
        diverted=("DIVERTED", "sum")
    )
)


Missing essential values:
FL_DATE               0
MKT_UNIQUE_CARRIER    0
ORIGIN_AIRPORT_ID     0
DEST_AIRPORT_ID       0
FLIGHTS               0
dtype: int64

Flight count values:
FLIGHTS
1.0    1292785
Name: count, dtype: int64

Cancellation summary:


,records,scheduled_flights,cancelled,diverted
MKT_UNIQUE_CARRIER,,,,
AA,514206,514206.0,11726.0,1743.0
DL,418790,418790.0,4541.0,1141.0
WN,359789,359789.0,1967.0,946.0


Establish the validated records

In [5]:

# Only include meaningful scheduled flight records
assert flights[essential].notna().all().all(), "Missing essential fields: investigate before filtering"
assert (flights["FLIGHTS"] > 0).all(), "Non-positive scheduled counts: investigate before filtering"
clean = flights.dropna(subset=essential).copy()

clean = clean[clean["FLIGHTS"] > 0].copy()

# Remove impossible self-loops if any exist
self_loops = (
    clean["ORIGIN_AIRPORT_ID"]
    == clean["DEST_AIRPORT_ID"]
).sum()

print("Self-loop flight records:", self_loops)

# Do not silently discard self-loops: investigate if found
assert self_loops == 0, "Review origin=destination records"

print("Validated records:", f"{len(clean):,}")


Self-loop flight records: 0
Validated records: 1,292,785


### Coverage and duplicate validation (added for the revised proposal, Step 1)

The month assertions above do not prove that every calendar day is present. These cells check
that each airline has records on all 91 days from 1 April to 30 June 2025, and that no flight
record is repeated. They only inspect the data: `flights` and `clean` are not changed.

In [6]:
# Paths for validation tables (same Results folder used by Notebook 02)
VALIDATION_TABLES = BASE.parent / "Results" / "Tables"
VALIDATION_TABLES.mkdir(parents=True, exist_ok=True)

expected_days = pd.date_range("2025-04-01", "2025-06-30", freq="D")
assert len(expected_days) == 91

coverage_rows = []

for airline in AIRLINES:
    airline_flights = clean[clean["MKT_UNIQUE_CARRIER"] == airline]
    daily_counts = (
        airline_flights.groupby(airline_flights["FL_DATE"].dt.normalize())
        .size()
        .reindex(expected_days, fill_value=0)
    )
    missing_days = daily_counts[daily_counts == 0].index

    coverage_rows.append({
        "Airline": airline,
        "Expected days": len(expected_days),
        "Days with records": int((daily_counts > 0).sum()),
        "Missing days": len(missing_days),
        "First date": airline_flights["FL_DATE"].min().date(),
        "Last date": airline_flights["FL_DATE"].max().date(),
        "Min records per day": int(daily_counts.min()),
        "Median records per day": float(daily_counts.median()),
        "Max records per day": int(daily_counts.max())
    })

    assert len(missing_days) == 0, f"{airline} has missing days"

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

coverage_df.to_csv(
    VALIDATION_TABLES / "01_daily_coverage_validation.csv",
    index=False
)

print("PASS: Every airline has records on all 91 days.")

PASS: Every airline has records on all 91 days.


,Airline,Expected days,Days with records,Missing days,First date,Last date,Min records per day,Median records per day,Max records per day
0,AA,91,91,0,2025-04-01,2025-06-30,5047,5726.0,5921
1,DL,91,91,0,2025-04-01,2025-06-30,3757,4673.0,5021
2,WN,91,91,0,2025-04-01,2025-06-30,3347,4086.0,4270


In [7]:
# A repeated row would inflate an edge weight, so test two definitions of "duplicate":
# 1. every column identical;
# 2. same date, marketing carrier, flight number, origin and destination.
flight_key = [
    "FL_DATE",
    "MKT_UNIQUE_CARRIER",
    "MKT_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID"
]

duplicate_rows = []

for airline in AIRLINES:
    airline_flights = clean[clean["MKT_UNIQUE_CARRIER"] == airline]
    duplicate_rows.append({
        "Airline": airline,
        "Records": len(airline_flights),
        "Fully identical records": int(airline_flights.duplicated().sum()),
        "Repeated flight-key records": int(
            airline_flights.duplicated(subset=flight_key).sum()
        )
    })

duplicates_df = pd.DataFrame(duplicate_rows)
display(duplicates_df)

duplicates_df.to_csv(
    VALIDATION_TABLES / "01_duplicate_validation.csv",
    index=False
)

assert duplicates_df["Fully identical records"].sum() == 0
assert duplicates_df["Repeated flight-key records"].sum() == 0

print("PASS: No duplicate flight records.")

PASS: No duplicate flight records.


,Airline,Records,Fully identical records,Repeated flight-key records
0,AA,514206,0,0
1,DL,418790,0,0
2,WN,359789,0,0


5. Construct the three airline networks

Aggregate scheduled flights into edges

In [8]:

edges = (
    clean.groupby(
        [
            "MKT_UNIQUE_CARRIER",
            "ORIGIN_AIRPORT_ID",
            "DEST_AIRPORT_ID"
        ],
        as_index=False
    )
    .agg(frequency=("FLIGHTS", "sum"))
)

edges = edges.rename(columns={
    "MKT_UNIQUE_CARRIER": "airline",
    "ORIGIN_AIRPORT_ID": "source",
    "DEST_AIRPORT_ID": "target"
})

display(edges.head())

print("\nDirected routes per airline:")
print(edges.groupby("airline").size())



Directed routes per airline:
airline
AA    1808
DL    1506
WN    1792
dtype: int64


,airline,source,target,frequency
0,AA,10135,11057,182.0
1,AA,10136,11298,439.0
2,AA,10140,11298,653.0
3,AA,10140,12892,234.0
4,AA,10140,13930,178.0


Create the airport node table

In [9]:

origin_nodes = clean[
    ["ORIGIN_AIRPORT_ID", "ORIGIN",
     "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR"]
].copy()

origin_nodes.columns = [
    "airport_id", "code", "city", "state"
]

dest_nodes = clean[
    ["DEST_AIRPORT_ID", "DEST",
     "DEST_CITY_NAME", "DEST_STATE_ABR"]
].copy()

dest_nodes.columns = [
    "airport_id", "code", "city", "state"
]

nodes = pd.concat(
    [origin_nodes, dest_nodes],
    ignore_index=True
).drop_duplicates()

# Verify airport IDs have consistent metadata
conflicts = nodes.groupby("airport_id").agg(
    codes=("code", "nunique"),
    cities=("city", "nunique"),
    states=("state", "nunique")
)

assert (conflicts <= 1).all().all(), (
    "Some airport IDs have conflicting metadata"
)

nodes = (
    nodes.drop_duplicates(subset=["airport_id"])
    .sort_values("airport_id")
    .reset_index(drop=True)
)

print("Unique airport IDs:", len(nodes))
display(nodes.head())


Unique airport IDs: 272


,airport_id,code,city,state
0,10135,ABE,"Allentown/Bethlehem/Easton, PA",PA
1,10136,ABI,"Abilene, TX",TX
2,10140,ABQ,"Albuquerque, NM",NM
3,10141,ABR,"Aberdeen, SD",SD
4,10146,ABY,"Albany, GA",GA


Export six shared CSV files

In [10]:

for airline in AIRLINES:
    airline_edges = (
        edges[edges["airline"] == airline]
        [["source", "target", "frequency"]]
        .copy()
    )

    airport_ids = set(airline_edges["source"]) | set(
        airline_edges["target"]
    )

    airline_nodes = (
        nodes[nodes["airport_id"].isin(airport_ids)]
        .copy()
    )

    airline_edges.to_csv(
        PROCESSED / f"{airline}_edges.csv",
        index=False
    )

    airline_nodes.to_csv(
        PROCESSED / f"{airline}_nodes.csv",
        index=False
    )

    print(
        airline,
        "| airports:", len(airline_nodes),
        "| routes:", len(airline_edges)
    )


AA | airports: 230 | routes: 1808
DL | airports: 212 | routes: 1506
WN | airports: 104 | routes: 1792


6. Build and validate the NetworkX graphs

Construct directed weighted graphs

In [11]:

import networkx as nx

graphs = {}

for airline in AIRLINES:
    airline_nodes = pd.read_csv(
        PROCESSED / f"{airline}_nodes.csv"
    )
    airline_edges = pd.read_csv(
        PROCESSED / f"{airline}_edges.csv"
    )

    G = nx.DiGraph(airline=airline)

    for row in airline_nodes.itertuples(index=False):
        G.add_node(
            int(row.airport_id),
            code=row.code,
            city=row.city,
            state=row.state
        )

    for row in airline_edges.itertuples(index=False):
        G.add_edge(
            int(row.source),
            int(row.target),
            frequency=float(row.frequency)
        )

    graphs[airline] = G

    assert G.number_of_nodes() == len(airline_nodes)
    assert G.number_of_edges() == len(airline_edges)

    print(
        airline,
        "Nodes:", G.number_of_nodes(),
        "Edges:", G.number_of_edges(),
        "Directed:", G.is_directed()
    )


AA Nodes: 230 Edges: 1808 Directed: True
DL Nodes: 212 Edges: 1506 Directed: True
WN Nodes: 104 Edges: 1792 Directed: True


Produce the first network summary table

In [12]:

summary = []

for airline, G in graphs.items():
    summary.append({
        "Airline": airline,
        "Airports": G.number_of_nodes(),
        "Directed routes": G.number_of_edges(),
        "Density": nx.density(G),
        "Total scheduled flights": sum(
            data["frequency"]
            for _, _, data in G.edges(data=True)
        ),
        "Strongly connected components":
            nx.number_strongly_connected_components(G)
    })

summary_df = pd.DataFrame(summary)

display(summary_df.round(4))

summary_df.to_csv(
    PROCESSED / "network_summary.csv",
    index=False
)


,Airline,Airports,Directed routes,Density,Total scheduled flights,Strongly connected components
0,AA,230,1808,0.0343,514206.0,1
1,DL,212,1506,0.0337,418790.0,1
2,WN,104,1792,0.1673,359789.0,1


Final validation before distributing the data

In [13]:

raw_counts = (
    clean.groupby("MKT_UNIQUE_CARRIER")["FLIGHTS"]
    .sum()
    .sort_index()
)

aggregated_counts = (
    edges.groupby("airline")["frequency"]
    .sum()
    .sort_index()
)

aggregated_counts.index.name = "MKT_UNIQUE_CARRIER"

pd.testing.assert_series_equal(
    raw_counts,
    aggregated_counts,
    check_names=False
)

print("PASS: Flight frequencies match the raw data.")

for airline, G in graphs.items():
    assert nx.number_of_selfloops(G) == 0

    assert all(
        data["frequency"] > 0
        for _, _, data in G.edges(data=True)
    )

print("PASS: No self-loops or invalid edge weights.")
print("PASS: Graph construction is complete.")


PASS: Flight frequencies match the raw data.
PASS: No self-loops or invalid edge weights.
PASS: Graph construction is complete.


### Documented scope decisions

**Diverted flights.** Diverted flights stay in the network as part of the scheduled route. BTS
reports the scheduled destination in `DEST` and records the actual diversion airport in separate
diversion fields, which are not part of our extract. The check below counts how many routes
depend on diverted flights.

**Puerto Rico and US Virgin Islands.** The BTS domestic data include airports in these US
territories. The revised proposal defines nodes as BTS airports and sets no geographic
exclusion, so they remain in the networks. The check below lists them for transparency.

In [14]:
scope_rows = []
territory_rows = []

for airline in AIRLINES:
    airline_flights = clean[clean["MKT_UNIQUE_CARRIER"] == airline]

    all_routes = set(
        zip(airline_flights["ORIGIN_AIRPORT_ID"], airline_flights["DEST_AIRPORT_ID"])
    )
    not_diverted = airline_flights[airline_flights["DIVERTED"] == 0]
    routes_without_diverted = set(
        zip(not_diverted["ORIGIN_AIRPORT_ID"], not_diverted["DEST_AIRPORT_ID"])
    )

    airline_nodes = pd.read_csv(PROCESSED / f"{airline}_nodes.csv")
    territory_nodes = airline_nodes[airline_nodes["state"].isin(["PR", "VI"])]

    scope_rows.append({
        "Airline": airline,
        "Diverted flights retained": int(airline_flights["DIVERTED"].sum()),
        "Routes existing only through diverted flights": len(all_routes - routes_without_diverted),
        "PR airports": int((territory_nodes["state"] == "PR").sum()),
        "VI airports": int((territory_nodes["state"] == "VI").sum())
    })

    for row in territory_nodes.itertuples(index=False):
        territory_rows.append({
            "Airline": airline,
            "AirportID": row.airport_id,
            "Code": row.code,
            "City": row.city,
            "State": row.state
        })

scope_df = pd.DataFrame(scope_rows)
territory_df = pd.DataFrame(territory_rows)

display(scope_df)
display(territory_df)

scope_df.to_csv(VALIDATION_TABLES / "01_scope_summary.csv", index=False)
territory_df.to_csv(VALIDATION_TABLES / "01_territory_airports.csv", index=False)

,Airline,Diverted flights retained,Routes existing only through diverted flights,PR airports,VI airports
0,AA,1743,0,1,2
1,DL,1141,0,1,2
2,WN,946,0,1,0


,Airline,AirportID,Code,City,State
0,AA,14843,SJU,"San Juan, PR",PR
1,AA,15024,STT,"Charlotte Amalie, VI",VI
2,AA,15027,STX,"Christiansted, VI",VI
3,DL,14843,SJU,"San Juan, PR",PR
4,DL,15024,STT,"Charlotte Amalie, VI",VI
5,DL,15027,STX,"Christiansted, VI",VI
6,WN,14843,SJU,"San Juan, PR",PR
